# 01 — Data understanding (Bronze profiling)

**DAT535 — OPDI air-traffic pipeline, group 08.** Exploration of the Bronze tables, before any cleaning. The findings
below drive the Silver design (missing-value forms, timestamp format, event types, content of the `info` JSON, altitude
unit, referential integrity between events and flights). Run it after the `bronze` layer
(`python projet_opdi/run_pipeline.py --env prod --layer ingest,bronze`).

In [ ]:
import json
import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parent / "src"))      # projet_opdi/src
import opdi_pipeline                                     # first: on the VM, findspark points to Spark 3.5 (/opt/spark)
from opdi_pipeline.config import load_settings
from opdi_pipeline.spark import get_spark

ENV = "prod"                     # "prod": full scope (~/opdi-data) | "dev": 10-day sample (~/opdi-data-dev)
settings = load_settings(ENV)
spark = get_spark(settings)
print(f"{ENV}: scope [{settings.start}, {settings.end}[ | data {settings.data_root} | Spark {spark.version}")

## Profiling of the Bronze tables

In the Silver layer, our objective is to transform raw Bronze tables into clean, validated, standardized, deduplicated, and enriched datasets ready for analytics and Gold aggregations.

Before implementing transformations, we profile the Bronze tables to evaluate:
- Volume and schema compatibility across tables
- Null rates on key flight attributes (`adep`, `ades`, `first_seen`, `last_seen`, `flt_id`, `typecode`)
- Temporal anomalies (such as inverted timestamps `first_seen > last_seen`)
- Duplicate keys and referential integrity between `flight_events` and `flight_list`

In [ ]:
from pyspark.sql.functions import (avg, col, desc, explode, from_json, lower, map_keys, to_timestamp, trim, when)
from pyspark.sql.functions import sum as spark_sum
from pyspark.sql.types import MapType, StringType

bronze_flights_df  = spark.read.parquet(str(settings.bronze_dir / "flight_list"))
bronze_events_df   = spark.read.parquet(str(settings.bronze_dir / "flight_events"))
bronze_airports_df = spark.read.parquet(str(settings.bronze_dir / "airports"))
bronze_runways_df  = spark.read.parquet(str(settings.bronze_dir / "runways"))

flights_count = bronze_flights_df.count()
events_count  = bronze_events_df.count()
print(f"Bronze flight_list   : {flights_count:>12,} rows")
print(f"Bronze flight_events : {events_count:>12,} rows")
print(f"Bronze airports      : {bronze_airports_df.count():>12,} rows")
print(f"Bronze runways       : {bronze_runways_df.count():>12,} rows")

# A value is "missing" if it is NULL, an empty string or the text "nan" (the three forms exist in OPDI)
def missing(c):
    return col(c).isNull() | (trim(col(c)) == "") | (lower(trim(col(c))) == "nan")

# 1) Missing values in flight_list
flight_cols = ["id", "icao24", "flt_id", "first_seen", "last_seen", "adep", "ades", "typecode", "icao_operator"]
res = bronze_flights_df.select([spark_sum(missing(c).cast("int")).alias(c) for c in flight_cols]).first().asDict()
print("\nMissing values in flight_list (NULL, '' or 'nan'):")
for c in flight_cols:
    print(f"  {c:15s} {res[c]:>10,}  ({res[c] / flights_count * 100:5.2f} %)")

# 2) Timestamps are strings in Bronze: look at their format before casting them
print("\nTimestamp examples:", bronze_flights_df.select("first_seen").first()[0], "|",
      bronze_events_df.select("event_time").first()[0])

# 3) Temporal anomalies and duplicate keys in flight_list
inversions = bronze_flights_df.filter(to_timestamp("first_seen") > to_timestamp("last_seen")).count()
dup_ids = flights_count - bronze_flights_df.select("id").distinct().count()
print(f"Flights with first_seen > last_seen: {inversions:,}")
print(f"Duplicate flight ids               : {dup_ids:,}")

# 4) Event types: exact values used later in Silver and Gold
print("\nEvent types:")
bronze_events_df.groupBy("type").count().orderBy(desc("count")).show(40, truncate=False)

# 5) Keys of the JSON column `info`, per event type (sample of ~100,000 events)
sample_fraction = min(1.0, 100_000 / max(events_count, 1))
print(f"Keys found in the JSON column `info` (sample: {sample_fraction:.2%} of the events):")
(bronze_events_df.sample(fraction=sample_fraction, seed=42)
    .select("type", explode(map_keys(from_json("info", MapType(StringType(), StringType())))).alias("info_key"))
    .groupBy("type", "info_key").count()
    .orderBy("type", "info_key")
    .show(100, truncate=False))
example = bronze_events_df.filter(col("type") == "entry-runway").select("info").first()
print("Example of `info` for an 'entry-runway' event:", example[0] if example else None)

# 6) Altitude unit and anomalies (FL100 = 10,000 ft = 3,048 m), missing `info`
alt = bronze_events_df.agg(
    avg(when(col("type") == "first-xing-fl100", col("altitude"))).alias("fl100"),
    spark_sum((col("altitude") < 0).cast("int")).alias("negative"),
    spark_sum(missing("info").cast("int")).alias("info_missing"),
).first()
if alt["fl100"] is not None:
    print(f"\nMean altitude of 'first-xing-fl100' events: {alt['fl100']:,.0f}  (~10,000 if feet, ~3,048 if metres)")
print(f"Events with a negative altitude: {alt['negative']:,} | events without `info`: {alt['info_missing']:,}")

# 7) Referential integrity: do the flights referenced by events exist in flight_list?
event_flights = bronze_events_df.select("flight_id").distinct()
n_event_flights = event_flights.count()
matched = event_flights.join(bronze_flights_df.select(col("id").alias("flight_id")), "flight_id", "left_semi").count()
print(f"\nDistinct flights in flight_events: {n_event_flights:,} | found in flight_list: {matched:,}"
      f" ({matched / max(n_event_flights, 1) * 100:.2f} %)")